In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Automatically discover candidate mappings for any network

This notebook creates provisional candidate mappings for `NETWORK_ID` from `meta_vars`. It uses variable names, standard names, descriptions, and units to find likely candidates for the six canonical variables.

Automatic discovery is a screening step, not final semantic validation. Review the candidates and their aggregation meaning before marking mappings as accepted.

## 1. Setup

In [17]:
import sys
function_path = '../func/'
sys.path.append(function_path)

In [18]:
from IPython.display import display
import sqlalchemy as sa
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path
from func_variable_mapping import (
    CANONICAL_VARIABLES, CANONICAL_RULES, save_candidate_mappings,
)


HERE = Path.cwd()
OUTPUT_DIR = HERE / 'outfile'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(HERE.parent / 'func'))

DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    '&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass',
)
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
NETWORK_ID = 5
MIN_SCORE = 3

## 2. Load the selected network's variable catalog

In [19]:
catalog_query = sa.text("""
SELECT vars_id, network_id, net_var_name::text AS net_var_name, unit,
       standard_name, cell_method, display_name, short_name,
       long_description
FROM meta_vars
WHERE network_id = :network_id
ORDER BY standard_name
""")

network_catalog = pd.read_sql(
    catalog_query, engine, params={'network_id': NETWORK_ID}
)
display(network_catalog)

,vars_id,network_id,net_var_name,unit,standard_name,cell_method,display_name,short_name,long_description
0,817,5,BarometricPressure,millibar,air_pressure,time: point,Air Pressure (Point),air_pressure_point,Atmospheric pressure
1,569,5,Tx_Climatology,celsius,air_temperature,t: maximum within days t: mean within months t...,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean ...,Climatological mean of monthly mean maximum da...
2,463,5,MIN_TEMP,celsius,air_temperature,time: minimum,Temperature (Min.),air_temperature_minimum,Minimum daily temperature
3,571,5,T_mean_Climatology,celsius,air_temperature,t: mean within days t: mean within months t: m...,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean wit...,Climatological mean of monthly mean of mean da...
4,1328,5,air_temp_1,celsius,air_temperature,time: point,Temperature (Mean),T,NaN
5,1329,5,air_temp_2,celsius,air_temperature,time: point,Temperature (Mean),T,NaN
6,657,5,AirTemp,celsius,air_temperature,time: point,Temperature (Point),air_temperature_point,Hourly air temperature instantaneous
7,570,5,Tn_Climatology,celsius,air_temperature,t: minimum within days t: mean within months t...,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean ...,Climatological mean of monthly mean minimum da...
8,464,5,MAX_TEMP,celsius,air_temperature,time: maximum,Temperature (Max.),air_temperature_maximum,Maximum daily temperature
9,672,5,HFT3_1_Avg,W m-2,downward_heatflux_in_soil,time: mean,Soil Heatflux 1,soil_heatflux_1,soil heat flux ( 1 of 2)


## 3. Define transparent discovery rules

Positive patterns add evidence; exclusion patterns prevent common false matches. Adjust these expressions if the selected network uses unusual abbreviations.

In [32]:
DISCOVERY_RULES = {
    'air_temperature': {
        'include': [r'air.*temp', r'temp.*air', r'(^|_)temperature($|_)'],
        'exclude': [r'min', r'max', r'dew', r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'mean',
    },
    'daily_min_temperature': {
        'include': [r'min(imum)?.*temp', r'temp.*min(imum)?'],
        'exclude': [r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'min',
    },
    'daily_max_temperature': {
        'include': [r'max(imum)?.*temp', r'temp.*max(imum)?'],
        'exclude': [r'soil', r'water', r'road', r'surface', r'climatology'],
        'daily_aggregation': 'max',
    },
    'precipitation_amount': {
        'include': [r'precip', r'pcpn', r'(^|_)rain($|_)', r'rainfall'],
        'exclude': [r'snow', r'snw', r'climatology'],
        'daily_aggregation': 'sum',
    },
    'snowfall_amount': {
        'include': [r'snow.*fall', r'snwfl', r'standard.*snow'],
        'exclude': [r'depth', r'dpth', r'height', r'climatology'],
        'daily_aggregation': 'sum',
    },
    'snow_depth': {
        'include': [
            r'snow.*depth', r'depth.*snow',
            r'height.*snow', r'surface.*snow.*thickness',

        ],
        'exclude': [r'fall', r'amount', r'climatology'],
        'daily_aggregation': 'fixed_hour',
    },
}

## 4. Score and rank candidates

In [33]:
def normalized_text(row, columns):
    return ' '.join(str(row.get(column) or '') for column in columns).lower()


def score_candidate(row, canonical_name):
    rule = DISCOVERY_RULES[canonical_name]
    name_text = normalized_text(
        row, ['net_var_name', 'short_name', 'display_name'])
    metadata_text = normalized_text(
        row, ['standard_name', 'long_description', 'cell_method'])
    all_text = name_text + ' ' + metadata_text

    if any(re.search(pattern, all_text) for pattern in rule['exclude']):
        return 0, 'excluded keyword'

    name_hits = sum(bool(re.search(pattern, name_text))
                    for pattern in rule['include'])
    metadata_hits = sum(bool(re.search(pattern, metadata_text))
                        for pattern in rule['include'])
    score = 3 * name_hits + 3 * metadata_hits

    expected_unit = CANONICAL_RULES[canonical_name].unit.lower()
    actual_unit = str(row.get('unit') or '').lower()
    unit_tokens = {
        'celsius': ['celsius', 'degc', 'degree_c', 'degrees_c', '°c'],
        'mm': ['mm', 'millimet'],
        'cm': ['cm', 'centimet'],
    }[expected_unit]
    unit_match = any(token in actual_unit for token in unit_tokens)
    if unit_match:
        score += 1

    evidence = f'name hits={name_hits}; metadata hits={metadata_hits}; unit match={unit_match}'
    return score, evidence


rows = []
for _, variable in network_catalog.iterrows():
    for canonical_name in CANONICAL_VARIABLES:
        score, evidence = score_candidate(variable, canonical_name)
        if score >= MIN_SCORE:
            rule = DISCOVERY_RULES[canonical_name]
            rows.append({
                'vars_id': int(variable['vars_id']),
                'net_var_name': variable['net_var_name'],
                'canonical_variable': canonical_name,
                'unit': variable['unit'],
                'daily_aggregation': rule['daily_aggregation'],
                'source_family': f'network{NETWORK_ID}_auto',
                'discovery_score': score,
                'discovery_evidence': evidence,
                'mapping_status': 'needs_review',
            })

discovered_candidates = pd.DataFrame(rows)
if not discovered_candidates.empty:
    discovered_candidates = discovered_candidates.sort_values(
        ['canonical_variable', 'discovery_score', 'vars_id'],
        ascending=[True, False, True],
    )
    discovered_candidates['priority'] = (
        discovered_candidates.groupby('canonical_variable').cumcount() + 1
    )
display(discovered_candidates)

,vars_id,net_var_name,canonical_variable,unit,daily_aggregation,source_family,discovery_score,discovery_evidence,mapping_status,priority
3,657,AirTemp,air_temperature,celsius,mean,network5_auto,16,name hits=3; metadata hits=2; unit match=True,needs_review,1
1,1328,air_temp_1,air_temperature,celsius,mean,network5_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,2
2,1329,air_temp_2,air_temperature,celsius,mean,network5_auto,7,name hits=1; metadata hits=1; unit match=True,needs_review,3
5,675,L_down_corr_Avg,air_temperature,W m-2,mean,network5_auto,3,name hits=0; metadata hits=1; unit match=False,needs_review,4
14,677,L_up_corr_Avg,air_temperature,W m-2,mean,network5_auto,3,name hits=0; metadata hits=1; unit match=False,needs_review,5
4,464,MAX_TEMP,daily_max_temperature,celsius,max,network5_auto,13,name hits=2; metadata hits=2; unit match=True,needs_review,1
0,463,MIN_TEMP,daily_min_temperature,celsius,min,network5_auto,13,name hits=2; metadata hits=2; unit match=True,needs_review,1
7,1332,cum_pcpn_amt,precipitation_amount,mm,sum,network5_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,1
9,1333,pcpn_amt_pst1hr,precipitation_amount,mm,sum,network5_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,2
10,1334,pcpn_amt_pst24hrs,precipitation_amount,mm,sum,network5_auto,10,name hits=2; metadata hits=1; unit match=True,needs_review,3


## 5. Review and override

List false-positive IDs under `EXCLUDE`. Use `MANUAL_CANDIDATES` for variables missed by the patterns or to override inferred semantics. Re-run this cell after editing the lists.

In [34]:
# Exclusions are (vars_id, canonical_variable) pairs.
EXCLUDE = {
    (675, 'air_temperature'),
    (677, 'air_temperature')

}

# Add or replace reviewed candidates as vars_id: canonical_variable.
# Name, unit, aggregation, source, and priority are filled automatically.
MANUAL_CANDIDATES = {
    468: 'snow_depth',  # SNOW_ON_THE_GROUND is depth, not snowfall.
    # 123: 'air_temperature',
}

reviewed = discovered_candidates.copy()
if not reviewed.empty and EXCLUDE:
    excluded = pd.MultiIndex.from_tuples(EXCLUDE)
    row_keys = pd.MultiIndex.from_frame(
        reviewed[['vars_id', 'canonical_variable']])
    reviewed = reviewed.loc[~row_keys.isin(excluded)].copy()

if MANUAL_CANDIDATES:
    manual_rows = []
    for vars_id, canonical_name in MANUAL_CANDIDATES.items():
        if canonical_name not in CANONICAL_VARIABLES:
            raise ValueError(
                f'Unknown canonical variable for vars_id {vars_id}: '
                f'{canonical_name}'
            )
        catalog_match = network_catalog[
            network_catalog['vars_id'] == vars_id
        ]
        if catalog_match.empty:
            raise ValueError(
                f'vars_id {vars_id} is not in network {NETWORK_ID}'
            )
        variable = catalog_match.iloc[0]
        manual_rows.append({
            'vars_id': int(vars_id),
            'net_var_name': variable['net_var_name'],
            'canonical_variable': canonical_name,
            'unit': variable['unit'],
            'daily_aggregation': (
                DISCOVERY_RULES[canonical_name]['daily_aggregation']
            ),
            'source_family': f'network{NETWORK_ID}_reviewed',
            'discovery_score': np.nan,
            'discovery_evidence': 'manual reviewed mapping',
            'mapping_status': 'candidate',
        })

    manual = pd.DataFrame(manual_rows)
    manual_keys = set(zip(manual['vars_id'], manual['canonical_variable']))
    if not reviewed.empty:
        keep = [
            (row.vars_id, row.canonical_variable) not in manual_keys
            for row in reviewed.itertuples()
        ]
        reviewed = reviewed.loc[keep]
    reviewed = pd.concat([reviewed, manual], ignore_index=True)

# Manual candidates sort first within a canonical variable; priorities
# are regenerated so they never need to be entered by hand.
reviewed['_manual_order'] = np.where(
    reviewed['mapping_status'].eq('candidate'), 0, 1
)
reviewed = reviewed.sort_values(
    ['canonical_variable', '_manual_order', 'discovery_score', 'vars_id'],
    ascending=[True, True, False, True],
).reset_index(drop=True)
reviewed['priority'] = (
    reviewed.groupby('canonical_variable').cumcount() + 1
)
reviewed = reviewed.drop(columns='_manual_order')
display(reviewed)

,vars_id,net_var_name,canonical_variable,unit,daily_aggregation,source_family,discovery_score,discovery_evidence,mapping_status,priority
0,657,AirTemp,air_temperature,celsius,mean,network5_auto,16.0,name hits=3; metadata hits=2; unit match=True,needs_review,1
1,1328,air_temp_1,air_temperature,celsius,mean,network5_auto,7.0,name hits=1; metadata hits=1; unit match=True,needs_review,2
2,1329,air_temp_2,air_temperature,celsius,mean,network5_auto,7.0,name hits=1; metadata hits=1; unit match=True,needs_review,3
3,464,MAX_TEMP,daily_max_temperature,celsius,max,network5_auto,13.0,name hits=2; metadata hits=2; unit match=True,needs_review,1
4,463,MIN_TEMP,daily_min_temperature,celsius,min,network5_auto,13.0,name hits=2; metadata hits=2; unit match=True,needs_review,1
5,1332,cum_pcpn_amt,precipitation_amount,mm,sum,network5_auto,10.0,name hits=2; metadata hits=1; unit match=True,needs_review,1
6,1333,pcpn_amt_pst1hr,precipitation_amount,mm,sum,network5_auto,10.0,name hits=2; metadata hits=1; unit match=True,needs_review,2
7,1334,pcpn_amt_pst24hrs,precipitation_amount,mm,sum,network5_auto,10.0,name hits=2; metadata hits=1; unit match=True,needs_review,3
8,465,ONE_DAY_PRECIPITATION,precipitation_amount,mm,sum,network5_auto,7.0,name hits=1; metadata hits=1; unit match=True,needs_review,4
9,466,ONE_DAY_RAIN,precipitation_amount,mm,sum,network5_auto,7.0,name hits=1; metadata hits=1; unit match=True,needs_review,5


## 6. Build and save this network's candidate mappings

The diagnostic columns remain in `discovered_candidates`; the final registry contains the same operational columns as the network-2 registry. The output filename includes the network ID.

In [35]:
NETWORK_ID

5

In [36]:
mapping_columns = [
    'vars_id', 'net_var_name', 'canonical_variable',
    'daily_aggregation', 'priority', 'source_family', 'mapping_status',
]
candidate_mappings = (
    reviewed[mapping_columns]
    .sort_values(['canonical_variable', 'priority', 'vars_id'])
    .reset_index(drop=True)
)
# display(candidate_mappings)

# Include the network ID in the output filename.
candidate_mapping_path = (
    OUTPUT_DIR / f'0.net{NETWORK_ID}_candidate_mappings.csv'
)
all_saved_mappings = save_candidate_mappings(
    NETWORK_ID, candidate_mappings, candidate_mapping_path
)
print(f'Saved network {NETWORK_ID} mappings to {candidate_mapping_path}')
display(all_saved_mappings)

Saved network 5 mappings to /workspaces/crmprtd/Quanlity_Control/pipeline/network_5/outfile/0.net5_candidate_mappings.csv


,network_id,vars_id,net_var_name,canonical_variable,daily_aggregation,priority,source_family,mapping_status
0,5,657,AirTemp,air_temperature,mean,1,network5_auto,needs_review
1,5,1328,air_temp_1,air_temperature,mean,2,network5_auto,needs_review
2,5,1329,air_temp_2,air_temperature,mean,3,network5_auto,needs_review
3,5,464,MAX_TEMP,daily_max_temperature,max,1,network5_auto,needs_review
4,5,463,MIN_TEMP,daily_min_temperature,min,1,network5_auto,needs_review
5,5,1332,cum_pcpn_amt,precipitation_amount,sum,1,network5_auto,needs_review
6,5,1333,pcpn_amt_pst1hr,precipitation_amount,sum,2,network5_auto,needs_review
7,5,1334,pcpn_amt_pst24hrs,precipitation_amount,sum,3,network5_auto,needs_review
8,5,465,ONE_DAY_PRECIPITATION,precipitation_amount,sum,4,network5_auto,needs_review
9,5,466,ONE_DAY_RAIN,precipitation_amount,sum,5,network5_auto,needs_review


## 7. Use the registry in the canonical-variable notebook

In [37]:
# Example
CANONICAL_VARIABLE = 'precipitation_amount'
candidate_rows = candidate_mappings[
    candidate_mappings['canonical_variable'] == CANONICAL_VARIABLE
]
candidate_ids = candidate_rows['vars_id'].tolist()
display(candidate_rows)

,vars_id,net_var_name,canonical_variable,daily_aggregation,priority,source_family,mapping_status
5,1332,cum_pcpn_amt,precipitation_amount,sum,1,network5_auto,needs_review
6,1333,pcpn_amt_pst1hr,precipitation_amount,sum,2,network5_auto,needs_review
7,1334,pcpn_amt_pst24hrs,precipitation_amount,sum,3,network5_auto,needs_review
8,465,ONE_DAY_PRECIPITATION,precipitation_amount,sum,4,network5_auto,needs_review
9,466,ONE_DAY_RAIN,precipitation_amount,sum,5,network5_auto,needs_review
10,655,Precip,precipitation_amount,sum,6,network5_auto,needs_review
